# Farm and Cat Data Merge

This notebook merges the prepared farm dataset with cat occurrence data in order to build a combined dataset for further analysis.

The farm data include sheep-related indicators by year and state, while the cat data include occurrence counts and forest-status information. The goal is to create a unified dataset that captures both agricultural and cat-related patterns across Australian territories and for the country as a whole.

In this notebook, we:

1. load the prepared farm and cat datasets;
2. align their time coverage;
3. harmonize territory names;
4. aggregate cat occurrence measures by year and territory;
5. construct forest-related cat indicators;
6. add national-level totals for farm variables;
7. merge farm and cat data into one final dataset;
8. save the result as `farm_cat.csv`.

The resulting dataset will later be extended with economic and news variables.

In [1]:
import pandas as pd
import numpy as np

farm_df = pd.read_csv("../datasets/farm.csv")
cat_df = pd.read_csv("../datasets/cat_only_rt.csv")

In [2]:
# --------------------------------------------------
# Restrict time coverage to the overlapping period
# - cat data: from 1990 onward
# - farm data: up to 2022
# --------------------------------------------------
cat_df = cat_df[cat_df["year"] >= 1990]
farm_df = farm_df[farm_df["year"] <= 2022]

In [3]:
# Inspect territory naming in both datasets
cat_df["stateTerritory"].unique()

array(['Australian Capital Territory', 'New South Wales',
       'Northern Territory', 'Queensland', 'South Australia', 'Victoria',
       'Unknown1', 'Western Australia', 'Tasmania'], dtype=object)

In [4]:
farm_df["state"].unique()

array(['New South Wales', 'Northern Territory', 'Queensland',
       'South Australia', 'Tasmania', 'Victoria', 'Western Australia'],
      dtype=object)

In [5]:
# Harmonize territory column names
keys = ["year", "territory"]
cat_df = cat_df.rename(columns={"stateTerritory": "territory"})

In [6]:
# Add national-level totals to the cat dataset
# This duplicates each row and assigns territory = "All Australia", allowing later aggregation at the country level
cat_df = pd.concat(
    [cat_df, cat_df.assign(territory="All Australia")],
    ignore_index=True
)

In [7]:
# Aggregate total cat occurrence counts by year and territory
cats_total = (
    cat_df.groupby(keys, as_index=False)["occurrenceCount"]
    .sum()
    .rename(columns={"occurrenceCount": "cats_occurrence_total"})
)

In [9]:
# Create forest/non-forest cat indicators using 2013 classification
cats_2013 = (
    cat_df.pivot_table(
        index=keys,
        columns="forest2013Status",
        values="occurrenceCount",
        aggfunc="sum",
        fill_value=0
    )
    .rename(columns=lambda c: f"cats_2013_{c.replace('-', '_')}")
    .reset_index()
)

# Create forest/non-forest cat indicators using 2018 classification
cats_2018 = (
    cat_df.pivot_table(
        index=keys,
        columns="forest2018Status",
        values="occurrenceCount",
        aggfunc="sum",
        fill_value=0
    )
    .rename(columns=lambda c: f"cats_2018_{c.replace('-', '_')}")
    .reset_index()
)

In [10]:
# Merge total occurrence counts with both forest-status datasets
cats_state_year = (
    cats_total
    .merge(cats_2013, on=keys, how="left")
    .merge(cats_2018, on=keys, how="left")
)

In [11]:
# Create a unified forest indicator
cats_state_year["cats_forest"] = np.where(
    cats_state_year["year"] < 2018,
    cats_state_year["cats_2013_forest"],
    cats_state_year["cats_2018_forest"]
)

In [12]:
# Keep only the final cat indicators needed for merging
cats_state_year = cats_state_year[
    ["year", "territory", "cats_occurrence_total", "cats_forest"]
].copy()

In [13]:
# Identify numeric farm columns for aggregation
num_cols = [c for c in farm_df.columns if c not in ["year", "state"]]

# Create national-level farm totals for each year
farm_all = (
    farm_df.groupby("year", as_index=False)[num_cols]
    .sum()
    .assign(state="All Australia")
)

# Reorder columns to match the original farm dataset
farm_all = farm_all[farm_df.columns]

# Append national totals to the farm dataset
farm_df = pd.concat([farm_df, farm_all], ignore_index=True)

# Harmonize farm territory column name
farm_df = farm_df.rename(columns={"state": "territory"})

In [14]:
# Merge farm and cat data
farm_cat_merged = farm_df.merge(cats_state_year, on=keys, how="outer")

In [15]:
# Remove territories not used in the final analysis
farm_cat_merged = farm_cat_merged[
    ~farm_cat_merged["territory"].isin(["Australian Capital Territory", "Unknown1"])
]

In [16]:
farm_cat_merged

,year,territory,lambs,rams,ewes,lamb_sheep_shorn,sheep_flock,sheep_purchased,cats_occurrence_total,cats_forest
0,1990,All Australia,15931.0,818.0,33771.0,73652.0,70896.0,4915.0,217.0,104.0
2,1990,New South Wales,4073.0,206.0,8649.0,18393.0,17123.0,976.0,69.0,48.0
3,1990,Northern Territory,0.0,0.0,2.0,15.0,4.0,15.0,43.0,11.0
4,1990,Queensland,3856.0,157.0,8095.0,17240.0,17443.0,1023.0,17.0,9.0
5,1990,South Australia,2803.0,145.0,5276.0,11211.0,10952.0,941.0,26.0,9.0
...,...,...,...,...,...,...,...,...,...,...
278,2022,Queensland,1056.0,59.0,2039.0,3267.0,3706.0,153.0,78.0,16.0
279,2022,South Australia,3003.0,123.0,5280.0,9578.0,8842.0,452.0,150.0,14.0
280,2022,Tasmania,567.0,20.0,1029.0,1804.0,1795.0,55.0,118.0,45.0
281,2022,Victoria,1608.0,50.0,2272.0,4236.0,4239.0,548.0,183.0,116.0


In [17]:
farm_cat_merged.to_csv("../datasets/farm_cat.csv", index=False)